# Aula 10 · Runge e splines

Esta aula apresenta o [capítulo 10 do site](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/). A ideia central: **com pontos igualmente espaçados, o polinômio de grau alto oscila — e piora com mais pontos**. A saída é escolher bem os pontos ou, na prática, usar um polinômio de grau baixo por trecho: a spline.

**Ao fim da aula você consegue:**

1. mostrar o fenômeno de Runge e reconhecer as oscilações do polinômio de grau alto;
2. usar os nós de Chebyshev quando dá para escolher onde medir;
3. entender a spline cúbica (continuidade de valor, inclinação e curvatura);
4. interpolar por trechos, com retas e com splines cúbicas, usando as bibliotecas.

**Roteiro:** 🧩 · 1. Runge · 2. Chebyshev · 3. 🧑‍🏫 por trecho · 4. splines na prática · 5. outra área · 🎯 prática · 🧩 o drone · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Aviação — o plano de voo de um drone.**
>
> *Um drone de inspeção de linhas de energia voa a 30 m de altura e, no meio do
> caminho, sobe a 80 m para passar sobre um prédio. O piloto definiu a altitude em
> 13 pontos, um por minuto, e o software precisa de uma curva suave entre eles. Há
> duas regras: **nunca abaixo de 20 m** (as árvores) e **nunca acima de 120 m** (o
> limite da ANAC). O estagiário usou "o polinômio que passa por todos os pontos".
> "**Posso liberar o voo?**"*

No fim da aula, você confere as duas curvas — o polinômio e a spline — contra as
regras.

## 1. Mais pontos, polinômio pior

A função de Runge, $f(x) = 1/(1 + 25x^2)$, é uma "lombada" suave em $[-1, 1]$. A
célula 📦 tem ela e a `lagrange`.

📖 [capítulo 10 · Mais pontos, polinômio pior](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#mais-pontos-polinomio-pior)

In [ ]:
# 📦 dados prontos — só rode esta célula
def runge(x):
    return 1 / (1 + 25 * x**2)


def lagrange(xs, ys, x):
    soma = 0.0
    for i in range(len(xs)):
        L = 1.0
        for j in range(len(xs)):
            if j != i:
                L = L * (x - xs[j]) / (xs[i] - xs[j])
        soma = soma + ys[i] * L
    return soma

**✍️ Passo 1.** Para `n` em `[5, 9, 13, 17]`: crie `nos = np.linspace(-1, 1, n)` e imprima o maior erro, `np.max(abs(lagrange(nos, runge(nos), xs) - runge(xs)))`, com `xs = np.linspace(-1, 1, 401)`.

In [ ]:
# ✍️ passo 1

**Preveja:** com mais pontos, o erro cai?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**Sobe**: 0,44 · 1,05 · 3,66 · 14,4. Com 17 pontos, o polinômio passa exatamente
por todos e erra por 14 entre eles, numa função que nunca passa de 1.

</details>

**✍️ Passo 2.** Desenhe `runge(xs)` e o polinômio por 13 pontos, com os pontos em `"o"`.

In [ ]:
# ✍️ passo 2

**Preveja:** onde o polinômio escapa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Nas **pontas**: perto de $-1$ e de $1$, ele despenca para $-3{,}6$. No meio, onde
há pontos dos dois lados, ele se comporta. É o **fenômeno de Runge**.

📖 [capítulo 10 · Mais pontos, polinômio pior](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#mais-pontos-polinomio-pior)

</details>

> ⚠️ **Armadilha.** Conferir o polinômio **só nos pontos da tabela** não mostra nada: ele passa
exatamente por todos. O erro mora entre eles. Sempre desenhe a curva inteira.

## 2. Nós de Chebyshev

Quando dá para escolher onde medir, ponha mais pontos perto das pontas:
$x_k = \cos\!\big((2k+1)\pi / (2n)\big)$.

📖 [capítulo 10 · Nós de Chebyshev](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#nos-de-chebyshev)

### 🎯 Sua vez — Um nó de Chebyshev

Escreva `no_chebyshev(k, n)`, que devolve o nó $x_k$ para $n$ nós em $[-1, 1]$.

In [ ]:
def no_chebyshev(k, n):
    # sua solução aqui
    pass

In [ ]:
confere(no_chebyshev, [
    ((0, 3), 0.8660254037844387),
    ((1, 3), 0.0),
    ((2, 4), -0.3826834323650897),
], tol=1e-9)

<details>
<summary><b>💡 Dica</b></summary>

Uma linha com `np.cos` e `np.pi`.

</details>

**✍️ Passo 3.** Repita o passo 1 com os nós de Chebyshev (um array `np.zeros(n)` preenchido com a sua `no_chebyshev`).

In [ ]:
# ✍️ passo 3

**Preveja:** e agora, o erro cai com mais pontos?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cai: 0,40 · 0,17 · 0,069 · 0,033. Com os nós agrupados nas pontas, o polinômio
não tem espaço para escapar. Mas só serve quando se **escolhe** onde medir.

📖 [capítulo 10 · Nós de Chebyshev](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#nos-de-chebyshev)

</details>

## 3. Um polinômio por trecho

📖 [capítulo 10 · Um polinômio por trecho](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#um-polinomio-por-trecho)

> 🧰 **Comando novo: `np.interp`**
>
> `np.interp(x, xs, ys)` faz a interpolação **linear** na tabela (`xs` crescente), para
> um número ou um array de valores. Fora da tabela, repete o valor da ponta.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
print(np.interp(9, [0, 6, 12, 18], [24.0, 23.0, 30.0, 27.0]))
print(np.interp([3, 9, 15], [0, 6, 12, 18], [24.0, 23.0, 30.0, 27.0]))

**✍️ Passo 4.** Desenhe a spline linear da função de Runge com 13 pontos igualmente espaçados: `np.interp(xs, nos, runge(nos))`, junto com `runge(xs)`.

In [ ]:
# ✍️ passo 4

**Preveja:** a curva oscila como o polinômio?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não oscila: fica sempre entre os pontos vizinhos. Mas faz **bicos** — a
inclinação muda de repente em cada ponto.

</details>

### 🧑‍🏫 No quadro — a spline cúbica

Caderno de papel aberto. No quadro:

1. um polinômio de grau 3 em cada trecho;
2. passa pelos dois pontos do trecho;
3. nos pontos internos: mesma inclinação e mesma curvatura dos dois lados;
4. as duas condições das pontas;
5. o sistema tridiagonal que sai disso.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

Com $n$ pontos: $n - 1$ trechos, $4(n - 1)$ coeficientes, e as equações de valor,
inclinação e curvatura contínuos (mais as duas das pontas). O sistema é
**tridiagonal**: cada equação só envolve os vizinhos.

</details>

### 🎯 Sua vez — Tem bico?

Escreva `bico(xs, ys, i)`, que devolve a inclinação da reta à **direita** do ponto `i` menos a inclinação à **esquerda** dele — zero quer dizer que não há bico.

In [ ]:
def bico(xs, ys, i):
    # sua solução aqui
    pass

In [ ]:
confere(bico, [
    (([0, 1, 3], [1, 3, 2], 1), -2.5),
    (([0, 1, 2], [0, 2, 4], 1), 0.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Duas inclinações, `(ys[i] - ys[i-1]) / (xs[i] - xs[i-1])` e a do trecho seguinte.

</details>

## 4. Splines na prática

Temperatura em João Pessoa medida a cada 3 horas: 0, 3, ..., 21 h; 25,2; 24,6; 24,4;
27,8; 30,6; 30,9; 28,3; 26,4 °C.

📖 [capítulo 10 · Splines na prática](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#splines-na-pratica)

> 🧰 **Comando novo: `from scipy.interpolate import CubicSpline`**
>
> `CubicSpline(xs, ys)` monta a spline cúbica da tabela e devolve algo que se usa **como
> uma função**: `s = CubicSpline(xs, ys)` e depois `s(x)`, para um número ou um array.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
from scipy.interpolate import CubicSpline

s = CubicSpline([0, 1, 2, 3], [0.0, 1.0, 0.0, 1.0])
print(s(1.5), s([0.5, 2.5]))

**✍️ Passo 5.** Crie as duas listas da temperatura, a spline com `CubicSpline`, e desenhe a spline e o polinômio de Lagrange em `np.linspace(0, 21, 211)`, com as medições em `"o"`.

In [ ]:
# ✍️ passo 5

**Preveja:** o que o polinômio faz entre meia-noite e 3 h?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**Sobe** para 26,3 °C, embora as medições caiam de 25,2 para 24,6: um morro
inventado. A spline passa pelos mesmos pontos, suave, sem inventar nada.

📖 [capítulo 10 · Splines na prática](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#splines-na-pratica)

</details>

## 5. Mesmo método, outra área

**Cinema.** Uma câmera passa pelos pontos $(0,0)$, $(2,1)$, $(4,0)$, $(4,-2)$, $(2,-3)$ e
$(0,-1)$ nos instantes 0 a 5 s. O caminho volta para trás — não é o gráfico de uma
função. O truque: uma spline para $x(t)$ e outra para $y(t)$.

📖 [capítulo 10 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#mesmo-metodo-outra-area)

**✍️ Passo 6.** Monte `CubicSpline(t, x)` e `CubicSpline(t, y)` e desenhe `plt.plot(sx(tt), sy(tt))` com `tt = np.linspace(0, 5, 200)`.

In [ ]:
# ✍️ passo 6

**Preveja:** o caminho passa suave pelos seis pontos, mesmo fazendo a volta?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Passa: uma curva fechada quase em forma de gota, sem bicos. Em $t = 2{,}5$ s, a
câmera está em $(4{,}30;\ -0{,}99)$.

📖 [capítulo 10 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-runge-splines/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

A prática desta aula é o próprio problema, logo abaixo: conferir se uma curva respeita limites — o que sempre se faz com uma interpolação antes de usá-la.

## 🧩 Resolvendo o problema

> *"**Posso liberar o voo?**"* — o estagiário.

A célula 📦 tem o plano de voo e os limites.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Plano de voo de um drone de inspeção: altitude (m) em cada minuto. Ele voa a
# 30 m, sobe a 80 m para passar sobre um prédio (minutos 5 a 7) e volta a 30 m.
minutos = np.linspace(0, 12, 13)  # 0, 1, 2, ..., 12
altitude = np.array([30.0, 30.0, 30.0, 30.0, 30.0, 80.0, 80.0, 80.0,
                     30.0, 30.0, 30.0, 30.0, 30.0])
MINIMA = 20.0     # abaixo disso, o drone bate nas árvores
MAXIMA = 120.0    # acima disso, o voo é proibido (ANAC)
t_voo = np.linspace(0, 12, 1201)

### 🎯 Sua vez — O voo é seguro?

Escreva `voo_seguro(alturas, minima, maxima)`, que devolve `True` se **todas** as alturas estiverem entre `minima` e `maxima`, e `False` se alguma sair.

In [ ]:
def voo_seguro(alturas, minima, maxima):
    # sua solução aqui
    pass

In [ ]:
confere(voo_seguro, [
    (([30.0, 50.0, 80.0], 20.0, 120.0), True),
    (([30.0, -5.0, 80.0], 20.0, 120.0), False),
    (([30.0, 130.0], 20.0, 120.0), False),
])

<details>
<summary><b>💡 Dica</b></summary>

Um laço sobre as alturas; `return False` na primeira fora dos limites, `return True` depois do laço.

</details>

As duas curvas, contra os limites:

In [ ]:
curva_polinomio = lagrange(minutos, altitude, t_voo)
curva_spline = CubicSpline(minutos, altitude)(t_voo)
print("polinômio: de", np.min(curva_polinomio), "a", np.max(curva_polinomio), "m -> seguro?",
      voo_seguro(curva_polinomio, MINIMA, MAXIMA))
print("spline:    de", np.min(curva_spline), "a", np.max(curva_spline), "m -> seguro?",
      voo_seguro(curva_spline, MINIMA, MAXIMA))

plt.figure()
plt.plot(t_voo, curva_spline, label="spline")
plt.plot(minutos, altitude, "o", label="plano de voo")
plt.axhline(MINIMA, color="black")
plt.axhline(MAXIMA, color="black")
plt.xlabel("minuto")
plt.ylabel("altitude (m)")
plt.grid()
plt.legend()
plt.show()

<details>
<summary><b>▶ O que os números dizem</b></summary>

O polinômio de grau 12 passa por todos os 13 pontos do plano e, entre eles, manda o
drone para **-66 m** — debaixo da terra — e para **708 m**, seis
vezes o limite da ANAC. É Runge: a subida brusca do prédio faz o polinômio oscilar,
e as oscilações explodem nas pontas do voo.

A spline fica entre **24.5 e 84.4 m**: desce um pouco abaixo dos
30 m perto da subida (a curva "respira" antes de subir), mas respeita as duas regras.
O voo pode ser liberado — **com a spline**.

</details>

## 📋 A lista

Abra a [Lista 10](https://lacouth.github.io/metodos_telecom-site/listas/lista10/). O **Exercício 01** é à mão (✏️): spline linear e Lagrange
pelos mesmos três pontos. Comece por ele, no papel.

**a)** Qual a inclinação da spline linear de cada lado do ponto $(1, 3)$?

<details>
<summary><b>▶ Resposta</b></summary>

$+2$ à esquerda e $-0{,}5$ à direita: um bico.

</details>

Termine o exercício e siga para o **Exercício 02**, os nós de Chebyshev.

## 🚪 Antes de sair

**1.** Por que mais pontos igualmente espaçados pioram o polinômio interpolador?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque o polinômio de grau alto é obrigado a passar por todos os pontos e compensa com oscilações cada vez maiores entre eles, principalmente nas pontas (fenômeno de Runge).

</details>

**2.** O que a spline cúbica exige nas junções que a spline linear não exige?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Mesma inclinação e mesma curvatura dos dois lados: sem bicos e sem trancos.

</details>

**3.** Quando os nós de Chebyshev não resolvem?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Quando os pontos já vêm prontos (medições de hora em hora, um catálogo): não dá para escolher onde medir. Aí a saída é a spline.

</details>

## 🏠 Para casa

- Explique para um colega, com um desenho, por que o drone do polinômio "bateria no chão".
- Termine a [Lista 10](https://lacouth.github.io/metodos_telecom-site/listas/lista10/).
- A Unidade 6 começa com a área sob uma curva: e se a curva for uma tabela?